# Hyperparameter Tuning

**WICHTIG**
Dieses Notebook soll nicht nocheinmal komplet durchlaufen werden lassen. Denn das Tuning dauerte 45 Minuten!

## 1. Bibliotheken Import

In [7]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score, ConfusionMatrixDisplay, make_scorer
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from scipy.stats import loguniform, randint
import matplotlib.pyplot as plt
import json
from pathlib import Path
import joblib

## 2. Daten einlesen

In [8]:
train_f = np.load("../../Model_data/ML_Daten/features_train.npz")
X_train_feat = train_f["X"]
y_train_raw = train_f["y"]

test_f = np.load("../../Model_data/ML_Daten/features_test.npz")
X_test_feat = test_f["X"]
y_test_raw = test_f["y"]

le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train_raw)
y_test_encoded = le.transform(y_test_raw)
class_names = le.classes_

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_feat)
X_test_scaled = scaler.transform(X_test_feat)

print(f"Train: {X_train_scaled.shape}, Test: {X_test_scaled.shape}")
print(f"Klassen: {list(class_names)}")

Train: (16782, 165), Test: (4112, 165)
Klassen: [np.str_('Auto'), np.str_('Laufen'), np.str_('Lift'), np.str_('Roundkick'), np.str_('Treppe'), np.str_('Velo'), np.str_('Zug')]


## 3. Hyperparameter-Tuning des besten Modells

**Nich nochmals ausführen, ausser das Json ging verloren!!**

In [ ]:
param_dist = {
    "learning_rate":     loguniform(0.01, 0.3),
    "max_iter":          randint(200, 1000),
    "max_depth":         randint(4, 15),
    "max_leaf_nodes":    randint(15, 127),
    "min_samples_leaf":  randint(5, 50),
    "l2_regularization": loguniform(0.0001, 10.0)
}

base_model = HistGradientBoostingClassifier(class_weight="balanced", early_stopping=True, validation_fraction=0.1, n_iter_no_change=15)
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scorer = make_scorer(f1_score, average="macro")
random_search = RandomizedSearchCV(
    estimator=base_model,
    param_distributions=param_dist,
    n_iter=80,
    scoring=scorer,
    cv=folds,
    verbose=2,
    n_jobs=-1,
    random_state=42
)

search_f1 = random_search.fit(X_train_scaled, y_train_encoded)
search_params = random_search.best_params_
print("\nBeste Hyperparameter-Kombination:")
for param, value in search_params.items():
    print(f"{param}: {value:.4f}" if isinstance(value, float) else f"{param}: {value}")

#Daten Speichern
out_dir = Path("../../Model_data/ML_Daten")
out_dir.mkdir(parents=True, exist_ok=True)

payload = {
    "best_params": random_search.best_params_,
    "best_cv_score_macro_f1": float(random_search.best_score_),
}
with open(out_dir / "best_hyperparameters.json", "w", encoding="utf-8") as f:
    json.dump(payload, f, indent=2, ensure_ascii=False)
print("Gespeichert:", out_dir / "best_hyperparameters.json")

### Daten aus dem Json herausholen

In [6]:
data_dir = Path("../../Model_data/ML_Daten")

with open(data_dir / "best_hyperparameters.json", encoding="utf-8") as f:
    hp = json.load(f)
best_params = hp["best_params"]
# optional, falls vorhanden:
cv_score = hp.get("best_cv_score_macro_f1")
print("Train:", X_train_scaled.shape, "Test:", X_test_scaled.shape)
print("Klassen:", list(class_names))
print("best_params geladen:", best_params)
if cv_score is not None:
    print("Bestes CV-Score (macro F1):", cv_score)

Train: (16782, 165) Test: (4112, 165)
Klassen: [np.str_('Auto'), np.str_('Laufen'), np.str_('Lift'), np.str_('Roundkick'), np.str_('Treppe'), np.str_('Velo'), np.str_('Zug')]
best_params geladen: {'l2_regularization': 0.0094, 'learning_rate': 0.1766, 'max_depth': 6, 'max_iter': 885, 'max_leaf_nodes': 38, 'min_samples_leaf': 36}
Bestes CV-Score (macro F1): 0.9836


### Model in Joblib speichern

In [10]:
best_model = HistGradientBoostingClassifier(
    **best_params, 
    class_weight="balanced", 
    random_state=42, 
    early_stopping=True, 
    validation_fraction=0.1, 
    n_iter_no_change=15)

best_model.fit(X_train_scaled, y_train_encoded)
y_pred = best_model.predict(X_test_scaled)
print("\nTest-Set Performance:")
print(classification_report(y_test_encoded, y_pred, target_names=class_names))


artifact = {
    "model": best_model,
    "scaler": scaler,
    "label_encoder": le,
    "class_names": list(class_names),
    "best_params": best_params,
}
out_path = Path("../../Model_data/ML_Daten/final_model.joblib")
joblib.dump(artifact, out_path)
print("Gespeichert:", out_path)


Test-Set Performance:
              precision    recall  f1-score   support

        Auto       0.98      1.00      0.99       787
      Laufen       0.98      0.97      0.98       798
        Lift       0.92      0.91      0.91       207
   Roundkick       0.99      0.99      0.99        85
      Treppe       0.92      0.92      0.92       146
        Velo       0.99      0.99      0.99      1161
         Zug       0.98      0.97      0.98       928

    accuracy                           0.98      4112
   macro avg       0.97      0.97      0.97      4112
weighted avg       0.98      0.98      0.98      4112

Gespeichert: ..\..\Model_data\ML_Daten\final_model.joblib
